In [1]:
import requests
from datetime import datetime
import time
from dateutil.parser import isoparse  # if timestamps are ISO
import pandas as pd

In [2]:
import findspark
# initialize findspark with spark directory
findspark.init()
# import pyspark
import pyspark
# create spark context
sc = pyspark.SparkContext()
# create spark session
spark = pyspark.sql.SparkSession(sc)

In [3]:
# inspect spark session
spark

Helper functions

In [4]:
BASE_URL = "https://new-api.ceo.ca/api/get_spiels"


In [5]:
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (sentiment-research; contact: renzo.gevaert@telenet.be)"
})

def dt_to_ms(dt: datetime) -> int:
    return int(dt.timestamp() * 1000)

In [6]:
session

## Fetch one page of spiels for a channel

In [7]:
def fetch_spiels_page(channel: str, until_ms: int,
                      max_retries: int = 5,
                      timeout: int = 20) -> dict:
    """
    Call the get_spiels endpoint for a given channel (e.g. 'arg')
    and an 'until' timestamp in ms.

    This version is robust:
    - Retries on ReadTimeout
    - Exponential backoff
    - Logs errors
    - Returns an empty 'spiels' list if all retries fail
    """
    params = {
        "channel": channel,
        "load_more": "top",
        "until": until_ms,
    }

    for attempt in range(1, max_retries + 1):
        try:
            resp = session.get(
                BASE_URL,
                params=params,
                timeout=timeout
            )
            resp.raise_for_status()
            return resp.json()

        except requests.exceptions.ReadTimeout:
            wait = 2 ** attempt  # 2, 4, 8, 16, 32 seconds
            print(f"[{channel}] ReadTimeout on attempt {attempt}/{max_retries}, "
                  f"waiting {wait}s before retrying...")
            time.sleep(wait)

        except requests.exceptions.RequestException as e:
            print(f"[{channel}] Request failed: {e}. Retrying in 5s...")
            time.sleep(5)

    # If we reach here, all retries failed → return a safe empty page
    print(f"[{channel}] Giving up after {max_retries} retries. Returning empty page.")
    return {"spiels": []}


## Define period

In [8]:
START = datetime(2024, 1, 15)
END   = datetime(2025, 10, 15)

## Define retrieval function

In [9]:
def harvest_channel(channel: str):
    """
    Get all spiels in [START, END] for a CEO.ca channel, e.g. 'arg'.
    Returns a list of dicts.
    """
    # start from the end of your window
    until_ms = dt_to_ms(END.replace(hour=23, minute=59, second=59))
    all_rows = []
    done = False
    page = 0

    # for optional progress percentage
    total_range_sec = (END - START).total_seconds()

    while not done:
        page += 1

        data = fetch_spiels_page(channel, until_ms)
        spiels = data.get("spiels", [])

        # if we get no spiels, either we're at the start of history
        # or fetch_spiels_page gave up after retries
        if not spiels:
            print(f"[{channel}] No spiels returned at page {page}. "
                  f"Stopping for this channel.")
            break  # no more history

        # oldest spiel in this batch (for logging + next 'until')
        oldest_ts = datetime.fromtimestamp(spiels[-1]["timestamp"] / 1000.0)

        # simple progress estimate
        progressed_sec = max(0.0, (oldest_ts - START).total_seconds())
        progress = min(1.0, progressed_sec / total_range_sec) if total_range_sec > 0 else 0.0

        print(
            f"[{channel}] Page {page}, "
            f"oldest={oldest_ts.date()}, "
            f"count={len(spiels)}, "
            f"~{progress*100:.1f}% through window"
        )

        # convert & filter
        for s in spiels:
            ts = datetime.fromtimestamp(s["timestamp"] / 1000.0)

            # once we go older than START we can stop completely
            if ts < START:
                done = True
                break

            if START <= ts <= END:
                all_rows.append({
                    "channel": s.get("channel"),
                    "timestamp": ts,
                    "username": s.get("name"),
                    "spiel_id": s.get("spiel_id"),
                    "text": s.get("spiel"),
                    "score": s.get("score"),
                    "votes": s.get("votes"),
                })

        # prepare next loop: go just before the oldest spiel we saw
        until_ms = dt_to_ms(oldest_ts) - 1

        # be nice to their servers
        time.sleep(0.3)

    return all_rows


## Channels

In [10]:
tickers = pd.read_csv("../Data/tickers.csv")
tickers

,tickers,Company Name
0,ARG.TO,Amerigo Resources Ltd
1,AYA.TO,Aya Gold & Silver Inc
2,CG.TO,Centerra Gold Inc
3,CGC.V,Canadian Gold Corp
4,CKG.V,Chesapeake Gold Corp
5,DMX.V,District Metals Corp
6,DV.V,Dolly Varden Silver Corp
7,EDG.V,Endurance Gold Corp
8,ELO.TO,Eloro Resources Ltd
9,EQX,Equinox Gold Corp


Link channels to tickers

In [39]:
ticker_channels = {
    #"ARG.TO": "arg", #ok
    #"AYA.TO": "aya", #ok
    #"CG.TO": "cg", #ok
    #"CGC.V": "cgc", #ok
    #"CKG.V": "ckg", #ok
    #"DMX.V": "dmx", #ok
    #"DV.V": "dv", #ok
    #"EDG.V": "edg", #ok
    #"ELO.TO": "elo", #ok
    #"EQX": "eqx", #ok
    #"ESK.V": "esk", #ok
    #"FAS.V": "fas", #ok
    #"FDY.TO": "fdy", #ok
    #"FINX.V": "finx", #ok
    #"FNM.V": "fnm", #ok
    #"FOM.TO": "fom", #ok
    #"FURY.TO": "fury", #ok
    #"FVL.TO": "fvl", #ok
    "FWZ.V": "fwz"
}

## Do this for each ticker and create dataframe

In [40]:
all_spiels = []
for my_ticker, channel in ticker_channels.items():
    rows = harvest_channel(channel)
    # keep original ticker label if you want
    for r in rows:
        r["ticker"] = my_ticker
    all_spiels.extend(rows)

df = pd.DataFrame(all_spiels)
print(df.head())

[fwz] Request failed: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')). Retrying in 5s...
[fwz] Page 1, oldest=2025-10-15, count=50, ~100.0% through window
[fwz] Page 2, oldest=2025-10-15, count=50, ~100.0% through window
[fwz] Page 3, oldest=2025-10-15, count=50, ~100.0% through window
[fwz] Page 4, oldest=2025-10-15, count=50, ~100.0% through window
[fwz] Page 5, oldest=2025-10-15, count=50, ~100.0% through window
[fwz] Page 6, oldest=2025-10-15, count=50, ~100.0% through window
[fwz] Page 7, oldest=2025-10-15, count=50, ~100.0% through window
[fwz] Page 8, oldest=2025-10-15, count=50, ~100.0% through window
[fwz] Page 9, oldest=2025-10-15, count=50, ~100.0% through window
[fwz] Page 10, oldest=2025-10-15, count=50, ~100.0% through window
[fwz] Page 11, oldest=2025-10-15, count=50, ~100.0% through window
[fwz] Page 12, oldest=2025-10-15, count=50, ~100.0% through window
[fwz] Page 13, oldest=2025-10-15, count=50, ~100.0% through window
[fwz

In [41]:
df.head()

,channel,timestamp,username,spiel_id,text,score,votes,ticker
0,fwz,2025-10-08 19:07:06.812,@beartoothclp,f8c815812ff2,I get it but seems like they’d need to find a ...,NaN,0,FWZ.V
1,fwz,2025-10-08 19:28:02.665,@owning_mahowny,8b54d7800d4a,$fwz all time high.,11.0,11,FWZ.V
2,fwz,2025-10-09 20:23:53.028,@Kkk187,68f4d56fe6ad,rollercoaster lately.,0.0,2,FWZ.V
3,fwz,2025-10-10 01:38:31.588,@CautiousNow,06bb259317c2,Remember that they recently did an out of cycl...,3.0,2,FWZ.V
4,fwz,2025-10-10 08:21:45.794,@LOLOLO,fefa8eb4017b,sector selling totally constructed to exploit ...,4.0,2,FWZ.V


In [45]:
df["ticker"].value_counts()

ticker
FWZ.V    233560
Name: count, dtype: int64

In [43]:
df.dtypes

channel              object
timestamp    datetime64[ns]
username             object
spiel_id             object
text                 object
score               float64
votes                 int64
ticker               object
dtype: object

In [44]:
# Columns that are entirely NaN / None
empty_cols = df.columns[df.isna().all()]
print("All-null columns:", list(empty_cols))

# Quick check for object columns (often problematic)
obj_cols = df.select_dtypes(include=["object"]).columns
print("Object columns:", list(obj_cols))


All-null columns: []
Object columns: ['channel', 'username', 'spiel_id', 'text', 'ticker']


In [ ]:
# df = df.drop("user_id", axis=1)

In [46]:
sdf = spark.createDataFrame(df)

In [47]:
sdf.write.mode("overwrite").format("parquet").save("../CEO_CA/fwz_ceo_ca.parquet")

## Checking

In [38]:
spark_df = spark.read.parquet("../CEO_CA/eqx_ceo_ca.parquet")

In [39]:
pandas_df = spark_df.toPandas()

In [40]:
pandas_df["ticker"].value_counts()

ticker
EQX    229850
Name: count, dtype: int64